# 04 PEFT Fine-Tuning in Google Colab

This notebook fine-tunes `google/flan-t5-base` using LoRA for ScholarSynth AI. It is designed for Google Colab with GPU enabled.

Before running:

1. Open this notebook in Google Colab.
2. Go to `Runtime > Change runtime type > T4 GPU`.
3. Upload or place these files in Google Drive:
   - `finetune_train.jsonl`
   - `finetune_val.jsonl`
   - `finetune_test.jsonl`

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate datasets sentencepiece evaluate rouge-score bert-score

## 2. Imports and GPU Check

In [ ]:
from pathlib import Path
import os
import json

import torch
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Mount Google Drive and Configure Paths

Recommended Google Drive layout:

```text
MyDrive/ScholarSynthAI/
├── finetune_train.jsonl
├── finetune_val.jsonl
├── finetune_test.jsonl
└── models/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/ScholarSynthAI')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROJECT_DIR / 'finetune_train.jsonl'
VAL_PATH = PROJECT_DIR / 'finetune_val.jsonl'
TEST_PATH = PROJECT_DIR / 'finetune_test.jsonl'

OUTPUT_DIR = PROJECT_DIR / 'models' / 'flan_t5_lora'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project dir:', PROJECT_DIR)
print('Train exists:', TRAIN_PATH.exists())
print('Val exists:', VAL_PATH.exists())
print('Test exists:', TEST_PATH.exists())
print('Output dir:', OUTPUT_DIR)

## 4. Optional Manual Upload

Run this cell only if you did not place the JSONL files in Drive. After upload, copy files into `PROJECT_DIR`.

In [ ]:
# Optional fallback upload. Skip this cell if files already exist in Google Drive.
# from google.colab import files
# uploaded = files.upload()
# for name in uploaded.keys():
#     target = PROJECT_DIR / name
#     Path(name).replace(target)
#     print('Moved', name, 'to', target)

## 5. Load and Inspect Dataset

In [ ]:
required_files = [TRAIN_PATH, VAL_PATH, TEST_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing dataset files: {missing}')

dataset = load_dataset(
    'json',
    data_files={
        'train': str(TRAIN_PATH),
        'validation': str(VAL_PATH),
        'test': str(TEST_PATH),
    },
)

print(dataset)
train_preview = pd.DataFrame(dataset['train'][:5])
train_preview[['task', 'topic', 'instruction', 'output']]

In [ ]:
task_counts = pd.DataFrame(dataset['train'])['task'].value_counts().reset_index()
task_counts.columns = ['task', 'train_examples']
task_counts

## 6. Load Base Model and Tokenizer

In [ ]:
MODEL_NAME = 'google/flan-t5-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print('Loaded:', MODEL_NAME)

## 7. Format and Tokenize Examples

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 192

def format_source(example):
    return (
        f"Instruction: {example['instruction']}\n"
        f"Task: {example['task']}\n"
        f"Topic: {example['topic']}\n"
        f"Input:\n{example['input']}\n\n"
        "Answer:"
    )

def preprocess(example):
    source = format_source(example)
    target = example['output']

    model_inputs = tokenizer(
        source,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=target,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized = dataset.map(preprocess, remove_columns=dataset['train'].column_names)
tokenized

## 8. Configure LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q', 'v'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 9. Train LoRA Adapter

Start with 2 epochs. If training is too slow, reduce `num_train_epochs` to 1. If it is fast and stable, increase to 3.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## 10. Save Adapter and Tokenizer

In [ ]:
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print('Saved LoRA adapter to:', OUTPUT_DIR)
print('Files:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

## 11. Quick Inference Test

In [ ]:
base_for_inference = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
fine_tuned_model = PeftModel.from_pretrained(base_for_inference, str(OUTPUT_DIR))
fine_tuned_model = fine_tuned_model.to(device)
fine_tuned_model.eval()

test_prompt = """
Instruction: Generate a short literature review on the topic using the provided papers.
Task: literature_review
Topic: Retrieval-augmented generation for research assistants
Input:
Paper 1: Retrieval-augmented generation improves factual grounding in knowledge-intensive tasks.
Paper 2: Scientific summarization systems benefit from domain-specific adaptation.
Paper 3: Vector search helps retrieve relevant paper sections for downstream generation.

Answer:
"""

inputs = tokenizer(
    test_prompt,
    return_tensors='pt',
    truncation=True,
    max_length=MAX_INPUT_LENGTH,
)
inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=160,
        num_beams=4,
        no_repeat_ngram_size=3,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 12. Zip Adapter for Download

Download this zip and place it back in the local project under `models/flan_t5_lora/`.

In [ ]:
!cd "{OUTPUT_DIR.parent}" && zip -r flan_t5_lora.zip flan_t5_lora
print('Adapter zip:', OUTPUT_DIR.parent / 'flan_t5_lora.zip')

## 13. What To Bring Back Locally

After training, copy this folder back into your local project:

```text
models/flan_t5_lora/
```

Then run final evaluation locally for:

- `fine_tuned_lora`
- `rag_plus_lora`